In [1]:
import torch
import torch.nn as nn
import json
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

In [10]:
with open('champ_data.json', 'r') as f:
    champ_data = json.load(f)

with open('champ_names.json', 'r') as f:
    champ_names = json.load(f)

NUM_CHAMPIONS_PER_GAME = 10
EMBEDDING_DIM = 16
HIDDEN_DIM = 128

In [17]:
str_to_idx = dict(zip(champ_names, range(len(champ_names))))
str_to_idx['Masked'] = len(champ_names)  # Add a special token for masked slots

In [ ]:
encode = lambda name: str_to_idx[name] # takes a champion name and returns the corresponding index
decode = lambda idx: champ_names[idx] # takes index and returns the corresponding champion name

print(decode(encode('Ahri')))

Ahri


In [21]:
class ChampionDataset(Dataset):
    def __init__(self, data, encode_fn):
        self.data = data
        self.encode = encode_fn

    def __len__(self):
        return len(self.data) * NUM_CHAMPIONS_PER_GAME

    def __getitem__(self, idx):
        match_idx = idx // NUM_CHAMPIONS_PER_GAME
        champ_idx = idx % NUM_CHAMPIONS_PER_GAME
        current_data = self.data[match_idx]
        masked_data = current_data.copy()
        masked_data[champ_idx] = 'Masked' # Mask the current champion
        x = [self.encode(name) for name in masked_data]
        return torch.tensor(x), torch.tensor(self.encode(current_data[champ_idx]))

In [ ]:
class LeagueDraftModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(num_embeddings=TOTAL_CHAMPIONS, embedding_dim=EMBEDDING_DIM)
        self.position_embedding = nn.Embedding(num_embeddings=NUM_CHAMPIONS_PER_GAME, embedding_dim=EMBEDDING_DIM)
        self.lm = nn.Linear(EMBEDDING_DIM, HIDDEN_DIM)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        output = self.fc(lstm_out[-1])
        return output

[['Sion', 'Warwick', 'Lux', 'Kalista', 'Zilean', 'TahmKench', 'XinZhao', 'Fizz', 'Ashe', 'Milio'], ['Riven', 'RekSai', 'Ekko', 'Lucian', 'Nami', 'Shen', 'Sylas', 'Zoe', 'Senna', 'Karma'], ['Olaf', 'LeeSin', 'Vex', 'Draven', 'Janna', 'Kennen', 'Rengar', 'XinZhao', 'Taliyah', 'Maokai'], ['Anivia', 'Jax', 'Yone', 'Tristana', 'Nautilus', 'DrMundo', 'LeeSin', 'Naafiri', 'Velkoz', 'Thresh'], ['Gnar', 'Rengar', 'Xerath', 'Tristana', 'Lulu', 'Vayne', 'Ivern', 'Lux', 'Samira', 'Thresh']]
